# Behavior Sequence Transformer (BST) with the 23 selected features

This notebook trains the **BST** ranker: option 1 of the three sequence-model designs compared earlier. This is based on Q. Chen's BST Paper which was developed for Taobao's recommendation algorithm. https://arxiv.org/pdf/1905.06874

For every (session, candidate) pair:

1. **Append the candidate to the history.** The candidate product is placed after the user's most recent `HIST_LEN` actions, giving a sequence of `HIST_LEN + 1` tokens: `[history actions …, candidate]`.
2. **Encode the whole sequence.** The sequence goes through a Transformer encoder without a causal mask, so every token attends to every other. The candidate attends to the history, and the history items are also re-represented *in the presence of this candidate*. This is also what makes it expensive: the history is re-encoded once for every candidate, about 90 times per session.
3. **Score.** The Transformer's output at the candidate's position is concatenated with the candidate's own embedding, the **23 context features** and the session's query vector. An MLP turns this into one logit. A sigmoid of the logit is the predicted probability of engagement.

**Training is pointwise, as in the BST paper.** Each candidate row is one example, with target 1 if the user engaged with it (label > 0) and 0 otherwise. Batches are drawn at random from all training candidates, across sessions, and the loss is binary cross-entropy. With `GRADED_WEIGHTS = True`, positive examples are weighted by their graded gain (click 1, cart 3, order 7), so the model is pushed harder to rank orders highly, consistent with the graded NDCG@10 used for model selection. **All training sessions are used without sampling**.

**The 23 context features** are aligned with the LightGBM model, which we use a a "white-box" comparator (`model_artifacts/lightgbm_popularity_query_search/cached_merged_features/selected_features/`):

| Group | Features |
|---|---|
| Personalisation (7) | `exact_wid_last_seen_days`, `shop_share`, `shop_last_seen_days`, `title_cos_max`, `i2v_cos_mean`, `last_i2v_cos`, `shop_aff_slate_pct` |
| Current-query matching (7) | `name_query_match` (corrected token overlap), `brand_name_query_match`, `cate_name_1_query_match` … `cate_name_4_query_match`, `query_match_count` |
| Product, history and slate context (7) | `brand_aff`, `cate3_aff`, `last_same_brand`, `last_same_cate3`, `qhist_wid_cnt`, `title_cos_profile`, `cate3_slate_share` |
| Training-derived popularity (2) | `item_ctr_train`, `cate3_ctr_train` (out-of-fold) |

Once again, `position`, `item_impr_train` and `i2v_cos_max_slate_pct` are not used for interpretability and data leakage reasons.

**Inputs (read only):**
- the step_3_person_match_feature_engineering.ipynb backup: session chunks, product tables, item2vec and term2vec vectors, and the step_1_data_preprocessing.ipynb split;
- The feature cache from the LightGBM Model (named after step_5_build_run_lightgbm_model.ipynb), whose rows are in the same order as `lgbm_features_all`. Step 3 verifies this on every row.

**Outputs:**
- `processed/bst_literal/` (new): configuration, checkpoints, the best model, validation and test predictions, metrics and performance tests;
- `/content/part8_local`: local working files.

**Model selection.** The model is selected on **graded NDCG@10** on the validation split (gains 0, 1, 3, 7), aligned to the LightGBM trainng method, so the results can be compared with its validation result. Binary NDCG@10, MRR and P@1 are reported as well. The test split is scored once, after training.

**Cost.** Each epoch passes `HIST_LEN + 1` tokens through the Transformer for each of about 12.3 million training candidates. Use a GPU runtime. Step 6 prints the measured speed and an estimate of the time per epoch after the first few hundred batches. Checkpoints are written every `CHECKPOINT_EVERY` batches, so a runtime limit loses little work, and rerunning the notebook continues from the last checkpoint, within the epoch.

## Step 0 — Setup and settings

Use the **USER INPUT** cell to change hyperparameters and training settings. The defaults are a small BST, with 64-dimensional embeddings, 2 Transformer layers and 2 attention heads, trained on all training sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

RAW = Path("/content/drive/MyDrive/BT4222 Final/JDSearch/Raw Data")
cache = sorted((RAW / "product_chunks").glob("*.parquet"))
cols = [f"cate_id_{k}" for k in range(1, 5)] + [f"cate_name_{k}" for k in range(1, 5)]
chunks = (pd.read_parquet(f, columns=cols) for f in cache) if cache else \
         pd.read_csv(RAW / "product_meta_data.txt", sep="\t", dtype=str, usecols=cols, chunksize=500_000)

pairs = {k: set() for k in range(1, 5)}
for ch in chunks:
    for k in range(1, 5):
        pairs[k].update(map(tuple, ch[[f"cate_id_{k}", f"cate_name_{k}"]].dropna().drop_duplicates().to_numpy()))

rows = []
for k, p in pairs.items():
    df = pd.DataFrame(list(p), columns=["id", "name"])
    rows.append({"level": k, "distinct IDs": df["id"].nunique(), "distinct names": df["name"].nunique(),
                 "IDs with >1 name": int((df.groupby("id")["name"].nunique() > 1).sum()),
                 "names shared by >1 ID": int((df.groupby("name")["id"].nunique() > 1).sum())})
print(pd.DataFrame(rows).set_index("level"))

Mounted at /content/drive
       distinct IDs  distinct names  IDs with >1 name  names shared by >1 ID
level                                                                       
1                76              76                 0                      0
2               671             630                 0                     25
3              5819            5399                30                    303
4              7137            6508                 0                    472


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================================================
#                                       USER INPUT
# ============================================================================================
HIST_LEN = 50                        # most recent history actions; the sequence has HIST_LEN + 1 tokens with the candidate
D_MODEL, N_HEADS, N_LAYERS = 64, 2, 2
DROPOUT = 0.1
MLP_HIDDEN = (256, 64)
BATCH_SIZE = 2048                    # candidates per training batch (drawn at random across sessions)
EVAL_BATCH_SIZE = 8192
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-5
GRADED_WEIGHTS = True                # weight positives by their gain (click 1, cart 3, order 7)
MAX_EPOCHS, PATIENCE = 10, 2         # early stopping on validation graded NDCG@10
CHECKPOINT_EVERY = 2000              # batches between mid-epoch checkpoints
TRAIN_SESSION_FRACTION = 1.0         # full training: all training sessions
RESUME = True
SAVE_ASSEMBLED_TO_DRIVE = False
# ============================================================================================
CONTEXT_FEATURES = ["exact_wid_last_seen_days", "shop_share", "shop_last_seen_days", "title_cos_max", "i2v_cos_mean", "last_i2v_cos", "shop_aff_slate_pct", "name_query_match", "brand_name_query_match", "cate_name_1_query_match", "cate_name_2_query_match", "cate_name_3_query_match", "cate_name_4_query_match", "query_match_count", "brand_aff", "cate3_aff", "last_same_brand", "last_same_cate3", "qhist_wid_cnt", "title_cos_profile", "cate3_slate_share", "item_ctr_train", "cate3_ctr_train"]     # step_5_build_run_lightgbm_model.ipynb's 23 features (fixed for this notebook)

In [ ]:
import gc, json, math, os, shutil, time, difflib
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as Fnn

try:
    import psutil
    def ram(label=""):
        print(f"[RAM] {label}: {psutil.Process().memory_info().rss / 1e9:.2f} GB in use")
except ImportError:
    def ram(label=""):
        pass

PROJECT_DIR = Path("/content/drive/MyDrive/BT4222 Final/JDSearch")
OUTPUT_DIR = PROJECT_DIR / "processed"
BACKUP_DIR = OUTPUT_DIR / "part3_features" / "intermediates_backup"   # read only
FEATURES_DIR = PROJECT_DIR / "model_artifacts" / "lightgbm_popularity_query_search" / "cached_merged_features" / "selected_features"   # step_5_build_run_lightgbm_model.ipynb's 23-feature cache (read only)
OUT_DIR = OUTPUT_DIR / "bst_literal"                                 # outputs of this notebook (new)
LOCAL_DIR = Path("/content/part8_local")                             # local working copies (new)
SRC_DIR, ASM_DIR = LOCAL_DIR / "src", LOCAL_DIR / "assembled"
for d in [OUT_DIR, SRC_DIR, ASM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
TYPE_CODE = {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}
TERM_SEP = "\x18"
TIME_BUCKETS_DAYS = [1 / 24, 1, 3, 7, 30, 90]        # bucket edges: <1h, <1d, <3d, <7d, <30d, <90d, older
BRAND, SHOP, CATE3 = 0, 1, 4                          # columns of prod_attr.npy
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE, "| torch", torch.__version__)

device: cuda | torch 2.14.1+cu130


## Step 1 — Gather the inputs

 This cell copies the needed files from the step_3_person_match_feature_engineering.ipynb backup to local disk and checks them. It also checks that step_5_build_run_lightgbm_model.ipynb's feature cache holds the three splits with all 23 features. Nothing is rebuilt here.

In [ ]:
NEEDED = ["sessions.parquet", "step1_stats.json", "arrays/lookup_wid.npy", "arrays/lookup_row.npy",
          "arrays/prod_attr.npy", "arrays/prod_meta.json", "arrays/i2v_keys.npy", "arrays/i2v_vecs.npy",
          "arrays/t2v_term_vecs.npy", "arrays/title_vocab.parquet"]


def fetch(rel):
    src, dst = BACKUP_DIR / rel, SRC_DIR / rel
    if not dst.exists():
        assert src.exists(), f"missing in the step_3_person_match_feature_engineering.ipynb backup: {rel} - rerun step_4_backup_feature_generation.ipynb Steps 0-3 to rebuild it"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    return dst


for rel in NEEDED:
    fetch(rel)
STATS = json.loads((SRC_DIR / "step1_stats.json").read_text())
N_CHUNKS = int(STATS["chunks"])
for i in range(N_CHUNKS):
    for k in ["cand", "hist"]:
        fetch(f"stage/{k}_{i:03d}.parquet")
STAGE = SRC_DIR / "stage"
n_rows = sum(pq.ParquetFile(STAGE / f"cand_{i:03d}.parquet").metadata.num_rows for i in range(N_CHUNKS))
assert n_rows == STATS["candidate_rows"], "session chunks are incomplete"

sessions = pd.read_parquet(SRC_DIR / "sessions.parquet").sort_values("session_id").reset_index(drop=True)
assert "split" in sessions.columns, "sessions.parquet has no split - rerun step_4_backup_feature_generation.ipynb Step 1"
LOOKUP_WID = np.load(SRC_DIR / "arrays/lookup_wid.npy", mmap_mode="r")
LOOKUP_ROW = np.load(SRC_DIR / "arrays/lookup_row.npy", mmap_mode="r")
PROD_ATTR = np.load(SRC_DIR / "arrays/prod_attr.npy", mmap_mode="r")
EMPTY_ROW = json.loads((SRC_DIR / "arrays/prod_meta.json").read_text())["n_prod"]
I2V_KEYS = np.load(SRC_DIR / "arrays/i2v_keys.npy")
I2V_VECS = np.load(SRC_DIR / "arrays/i2v_vecs.npy", mmap_mode="r")
T2V_TERMS = np.load(SRC_DIR / "arrays/t2v_term_vecs.npy", mmap_mode="r")
TITLE_INDEX = pd.Index(pd.read_parquet(SRC_DIR / "arrays/title_vocab.parquet")["term"])

for sp in ["train", "val", "test"]:
    f = FEATURES_DIR / f"{sp}.parquet"
    assert f.exists(), f"missing feature cache file: {f} - run step_5_build_run_lightgbm_model.ipynb's notebook (section 4) first"
    missing = set(CONTEXT_FEATURES + ["session_id", "wid"]) - set(pq.ParquetFile(f).schema_arrow.names)
    assert not missing, f"{sp}: feature cache lacks {missing}"
print(f"{N_CHUNKS} chunks, {n_rows:,} candidate rows, {len(sessions):,} sessions; "
      f"split: {sessions['split'].value_counts().to_dict()}")
ram("inputs")

86 chunks, 15,348,049 candidate rows, 171,175 sessions; split: {'train': 136940, 'test': 17118, 'val': 17117}
[RAM] inputs: 0.75 GB in use


## Step 2 — Vocabularies and the context features

The context features are fixed to the 23 above. Each categorical input is mapped to a compact index (0 = padding, 1 = unknown or missing). Each context feature is log-transformed if heavy-tailed, then standardised, with statistics streamed from the **training** split of the feature cache. Missing values become 0, and a 0/1 indicator is added for each feature that can be missing.

In [ ]:
def prod_rows(wids):
    # Row of each product in the product tables (EMPTY_ROW if absent) - step_3_person_match_feature_engineering.ipynb code.
    wids = np.asarray(wids, dtype=np.int64)
    i = np.minimum(np.searchsorted(LOOKUP_WID, wids), len(LOOKUP_WID) - 1)
    return np.where(np.asarray(LOOKUP_WID[i]) == wids, np.asarray(LOOKUP_ROW[i]), EMPTY_ROW)


def compact(values, uniques):
    # Map raw IDs to 2.. via a sorted array of known values; unknown / missing (-1) -> 1.
    values = np.asarray(values)
    if len(uniques) == 0:
        return np.ones(len(values), np.int32)
    i = np.minimum(np.searchsorted(uniques, values), len(uniques) - 1)
    ok = (values != -1) & (uniques[i] == values)
    return np.where(ok, i + 2, 1).astype(np.int32)


ATTR_UNIQUES = {}
for name, j in [("brand", BRAND), ("shop", SHOP), ("cate3", CATE3)]:
    col = np.asarray(PROD_ATTR[:, j])
    ATTR_UNIQUES[name] = np.unique(col[col != -1])
N_ITEMS = len(I2V_KEYS) + 2
VOCAB = {"item": N_ITEMS, **{k: len(v) + 2 for k, v in ATTR_UNIQUES.items()}, "type": 6, "tbucket": len(TIME_BUCKETS_DAYS) + 3}
print("vocabulary sizes (incl. padding and unknown):", VOCAB)


def item_index(wids):
    i = np.minimum(np.searchsorted(I2V_KEYS, wids), len(I2V_KEYS) - 1)
    return np.where(I2V_KEYS[i] == wids, i + 2, 1).astype(np.int32)


def attr_indices(wids):
    a = np.asarray(PROD_ATTR[prod_rows(wids)])
    return {k: compact(a[:, j], ATTR_UNIQUES[k]) for k, j in [("brand", BRAND), ("shop", SHOP), ("cate3", CATE3)]}


# ---------------- context features: step_5_build_run_lightgbm_model.ipynb's 23 ----------------
CTX = list(CONTEXT_FEATURES)
assert len(CTX) == 23

CTX_STATS = {}
if CTX:
    pf = pq.ParquetFile(FEATURES_DIR / "train.parquet")
    acc = {f: dict(n=0, nan=0, s=0.0, ss=0.0, ls=0.0, lss=0.0, mx=-np.inf) for f in CTX}
    for b in pf.iter_batches(batch_size=1_000_000, columns=CTX):
        d = b.to_pandas()
        for f in CTX:
            x = d[f].to_numpy(np.float64); m = ~np.isnan(x); v = x[m]
            if not len(v):
                acc[f]["nan"] += int((~m).sum()); continue
            lv = np.sign(v) * np.log1p(np.abs(v))
            a = acc[f]
            a["n"] += len(v); a["nan"] += int((~m).sum()); a["s"] += v.sum(); a["ss"] += (v * v).sum()
            a["ls"] += lv.sum(); a["lss"] += (lv * lv).sum(); a["mx"] = max(a["mx"], np.abs(v).max())
    for f, a in acc.items():
        n = max(a["n"], 1)
        mean, sd = a["s"] / n, math.sqrt(max(a["ss"] / n - (a["s"] / n) ** 2, 1e-12))
        use_log = a["mx"] > 20 * sd
        if use_log:
            mean, sd = a["ls"] / n, math.sqrt(max(a["lss"] / n - (a["ls"] / n) ** 2, 1e-12))
        CTX_STATS[f] = {"log": bool(use_log), "mean": float(mean), "sd": float(sd), "has_nan": bool(a["nan"] > 0)}
CTX_MISS = [f for f in CTX if CTX_STATS[f]["has_nan"]]
CTX_DIM = len(CTX) + len(CTX_MISS)
print(f"{len(CTX)} context features ({len(CTX_MISS)} with missing-value indicators) -> context width {CTX_DIM}")
if CTX:
    display(pd.DataFrame(CTX_STATS).T.head(40))


def prepare_context(df):
    # Apply the training-set transformation to a DataFrame of context features -> float16 matrix.
    out = np.zeros((len(df), CTX_DIM), dtype=np.float32)
    for j, f in enumerate(CTX):
        x = df[f].to_numpy(np.float64); st = CTX_STATS[f]
        if st["log"]:
            x = np.sign(x) * np.log1p(np.abs(x))
        out[:, j] = np.nan_to_num((x - st["mean"]) / st["sd"], nan=0.0)
    for j, f in enumerate(CTX_MISS):
        out[:, len(CTX) + j] = df[f].isna().to_numpy()
    return np.clip(out, -20, 20).astype(np.float16)


json.dump({"CONTEXT_FEATURES": CTX, "CTX_STATS": CTX_STATS, "VOCAB": VOCAB, "HIST_LEN": HIST_LEN,
           "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS, "TIME_BUCKETS_DAYS": TIME_BUCKETS_DAYS},
          open(OUT_DIR / "config.json", "w"), indent=1)

vocabulary sizes (incl. padding and unknown): {'item': 859414, 'brand': 181096, 'shop': 249737, 'cate3': 5806, 'type': 6, 'tbucket': 9}
23 context features (14 with missing-value indicators) -> context width 37


,log,mean,sd,has_nan
exact_wid_last_seen_days,False,17.488232,48.460209,True
shop_share,False,0.093946,0.166205,True
shop_last_seen_days,False,23.443185,56.339675,True
title_cos_max,False,0.214565,0.205202,True
i2v_cos_mean,False,0.494484,0.131845,True
last_i2v_cos,False,0.551766,0.212031,True
shop_aff_slate_pct,False,0.541406,0.210613,True
name_query_match,True,0.927274,0.441349,False
brand_name_query_match,False,0.045422,0.208228,False
cate_name_1_query_match,True,0.001073,0.02725,False


## Step 3 — Assemble the input arrays

For each split it builds, as memory-mapped NumPy files on local disk:
- *per session:* the most recent `HIST_LEN` actions (most recent first: product, brand, shop, category, action type and time bucket), the query vector, and the session's position in the candidate arrays;
- *per candidate:* product, brand, shop and category indices, the label, the product ID and the prepared context features.

Rows follow the order of the feature cache, and the cell checks `session_id` and `wid` on every row.

In [ ]:
SPLITS = ["train", "val", "test"]
BUCKET_EDGES = np.array(TIME_BUCKETS_DAYS)


class SequentialReader:
    # Reads consecutive rows of a Parquet file, row group by row group (keeps one row group in memory).
    def __init__(self, path, columns):
        self.pf, self.columns, self.g, self.buf = pq.ParquetFile(path), columns, 0, None

    def take(self, n):
        parts, need = [], n
        while need > 0:
            if self.buf is None or len(self.buf) == 0:
                self.buf = self.pf.read_row_group(self.g, columns=self.columns).to_pandas(); self.g += 1
            parts.append(self.buf.iloc[:need]); self.buf = self.buf.iloc[need:]; need -= len(parts[-1])
        return pd.concat(parts, ignore_index=True)


def query_vectors(queries):
    out = np.zeros((len(queries), T2V_TERMS.shape[1]), np.float32)
    for k, q in enumerate(queries):
        if q:
            idx = TITLE_INDEX.get_indexer(q.split(TERM_SEP)); idx = idx[idx >= 0]
            if len(idx):
                v = np.asarray(T2V_TERMS[idx]).mean(0); nv = np.linalg.norm(v)
                out[k] = v / nv if nv > 0 else v
    return out


def assemble(split):
    d, saved = ASM_DIR / split, OUT_DIR / "assembled" / split
    if not (d / "DONE").exists() and (saved / "DONE").exists():
        shutil.copytree(saved, d, dirs_exist_ok=True)          # reuse arrays saved to Drive in an earlier session
    if (d / "DONE").exists():
        print(f"[{split}] already assembled"); return
    d.mkdir(parents=True, exist_ok=True)
    sp = sessions[sessions["split"] == split]
    n_s, n_c = len(sp), int(sp["slate_size"].sum())
    mm = lambda name, shape, dt: np.lib.format.open_memmap(d / f"{name}.npy", mode="w+", dtype=dt, shape=shape)
    H = {k: mm(f"h_{k}", (n_s, HIST_LEN), np.int32) for k in ["item", "brand", "shop", "cate3"]}
    H["type"] = mm("h_type", (n_s, HIST_LEN), np.int8); H["tbucket"] = mm("h_tbucket", (n_s, HIST_LEN), np.int8)
    S = {"session_id": mm("s_session_id", (n_s,), np.int32), "ptr": mm("s_ptr", (n_s + 1,), np.int64),
         "qvec": mm("s_qvec", (n_s, T2V_TERMS.shape[1]), np.float16), "hist_len": mm("s_hist_len", (n_s,), np.int16)}
    C = {k: mm(f"c_{k}", (n_c,), np.int32) for k in ["item", "brand", "shop", "cate3"]}
    C["label"] = mm("c_label", (n_c,), np.int8); C["wid"] = mm("c_wid", (n_c,), np.int64)
    CX = mm("c_ctx", (n_c, max(CTX_DIM, 1)), np.float16)
    reader = SequentialReader(FEATURES_DIR / f"{split}.parquet", ["session_id", "wid"] + CTX) if CTX else None
    si = ci = 0
    S["ptr"][0] = 0
    for i in range(N_CHUNKS):
        ids = sp.loc[sp["chunk"] == i, "session_id"].to_numpy()
        if not len(ids):
            continue
        cand = pd.read_parquet(STAGE / f"cand_{i:03d}.parquet", columns=["session_id", "wid", "label"])
        cand = cand[cand["session_id"].isin(ids)]
        hist = pd.read_parquet(STAGE / f"hist_{i:03d}.parquet", columns=["session_id", "wid", "type", "days"])
        hist = hist[hist["session_id"].isin(ids)]
        k_s, k_c = len(ids), len(cand)
        # ---- sessions ----
        S["session_id"][si:si + k_s] = ids
        counts = cand.groupby("session_id", sort=True).size().reindex(ids).to_numpy()
        S["ptr"][si + 1:si + k_s + 1] = ci + np.cumsum(counts)
        qs = sessions.set_index("session_id").loc[ids, "query"].fillna("").to_numpy(dtype=object)
        S["qvec"][si:si + k_s] = query_vectors(qs).astype(np.float16)
        # ---- history: most recent HIST_LEN actions, most recent first ----
        hs = hist["session_id"].to_numpy()
        rank_from_end = hist.groupby("session_id").cumcount(ascending=False).to_numpy()
        keep = rank_from_end < HIST_LEN
        rows = np.searchsorted(ids, hs[keep]); cols = rank_from_end[keep]
        hw = hist["wid"].to_numpy()[keep]
        block = {"item": np.zeros((k_s, HIST_LEN), np.int32)}
        block["item"][rows, cols] = item_index(hw)
        for k, v in attr_indices(hw).items():
            block[k] = np.zeros((k_s, HIST_LEN), np.int32); block[k][rows, cols] = v
        t = hist["type"].to_numpy()[keep].astype(np.int64)
        block["type"] = np.zeros((k_s, HIST_LEN), np.int8); block["type"][rows, cols] = np.where(t >= 0, t + 1, 5)
        tb = np.searchsorted(BUCKET_EDGES, hist["days"].to_numpy()[keep], side="right") + 1
        block["tbucket"] = np.zeros((k_s, HIST_LEN), np.int8); block["tbucket"][rows, cols] = tb
        for k in H:
            H[k][si:si + k_s] = block[k]
        S["hist_len"][si:si + k_s] = np.minimum(np.bincount(np.searchsorted(ids, hs), minlength=k_s), HIST_LEN)
        # ---- candidates ----
        cw = cand["wid"].to_numpy()
        C["wid"][ci:ci + k_c] = cw; C["label"][ci:ci + k_c] = cand["label"].to_numpy()
        C["item"][ci:ci + k_c] = item_index(cw)
        for k, v in attr_indices(cw).items():
            C[k][ci:ci + k_c] = v
        if reader is not None:
            f = reader.take(k_c)
            assert (f["session_id"].to_numpy() == cand["session_id"].to_numpy()).all() and \
                   (f["wid"].to_numpy() == cw).all(), f"[{split}] row order differs from lgbm_features_all in chunk {i}"
            CX[ci:ci + k_c] = prepare_context(f)
        si += k_s; ci += k_c
        del cand, hist
    assert si == n_s and ci == n_c
    for arr in [*H.values(), *S.values(), *C.values(), CX]:
        arr.flush()
    (d / "DONE").write_text(json.dumps({"sessions": n_s, "candidates": n_c}))
    if SAVE_ASSEMBLED_TO_DRIVE:
        shutil.copytree(d, OUT_DIR / "assembled" / split, dirs_exist_ok=True)
    print(f"[{split}] {n_s:,} sessions, {n_c:,} candidates assembled")


for s in SPLITS:
    assemble(s)
ram("after assembly")

[train] 136,940 sessions, 12,279,873 candidates assembled
[val] 17,117 sessions, 1,536,651 candidates assembled
[test] 17,118 sessions, 1,531,525 candidates assembled
[RAM] after assembly: 2.09 GB in use


## Step 4 — Candidate batches (pointwise, across sessions)

BST treats every candidate as a separate example. Each epoch shuffles all training candidates and cuts them into batches of `BATCH_SIZE`. A batch therefore mixes candidates from many sessions. For each candidate, its session's history is looked up and the candidate is appended to it.

The shuffle of each epoch is fixed by the seed and the epoch number, so a run resumed from a mid-epoch checkpoint continues with exactly the batches it had not yet seen.

In [ ]:
class CandidateData:
    def __init__(self, split):
        d = ASM_DIR / split
        load = lambda n: np.load(d / f"{n}.npy", mmap_mode="r")
        self.H = {k: load(f"h_{k}") for k in ["item", "brand", "shop", "cate3", "type", "tbucket"]}
        self.sid, self.ptr, self.qvec = load("s_session_id"), np.asarray(load("s_ptr")), load("s_qvec")
        self.C = {k: load(f"c_{k}") for k in ["item", "brand", "shop", "cate3", "label", "wid"]}
        self.ctx = load("c_ctx")
        self.size = np.diff(self.ptr)
        self.cand_session = np.repeat(np.arange(len(self.size)), self.size)       # session row of each candidate
        self.n = int(self.ptr[-1])

    def rows_of_sessions(self, sess_idx):
        return np.concatenate([np.arange(self.ptr[s], self.ptr[s + 1]) for s in sess_idx]) if len(sess_idx) else np.array([], np.int64)

    def tensors(self, rows):
        # One batch of candidate rows -> history [B, L] of each candidate's session + candidate features.
        rows = np.sort(rows)
        s = self.cand_session[rows]
        T = {f"h_{k}": torch.from_numpy(np.asarray(v[s]).astype(np.int64)) for k, v in self.H.items()}
        for k in ["item", "brand", "shop", "cate3"]:
            T[f"c_{k}"] = torch.from_numpy(np.asarray(self.C[k][rows]).astype(np.int64))
        T["label"] = torch.from_numpy(np.asarray(self.C["label"][rows]).astype(np.int64))
        T["ctx"] = torch.from_numpy(np.asarray(self.ctx[rows], np.float32))
        T["qvec"] = torch.from_numpy(np.asarray(self.qvec[s], np.float32))
        return {k: v.to(DEVICE, non_blocking=True) for k, v in T.items()}, rows


DATA = {s: CandidateData(s) for s in SPLITS}
rng0 = np.random.default_rng(SEED)
n_tr_sess = len(DATA["train"].sid)
TRAIN_SESS = np.arange(n_tr_sess) if TRAIN_SESSION_FRACTION >= 1.0 else \
    np.sort(rng0.choice(n_tr_sess, size=int(n_tr_sess * TRAIN_SESSION_FRACTION), replace=False))
TRAIN_ROWS = DATA["train"].rows_of_sessions(TRAIN_SESS) if TRAIN_SESSION_FRACTION < 1.0 else np.arange(DATA["train"].n)
BATCHES_PER_EPOCH = math.ceil(len(TRAIN_ROWS) / BATCH_SIZE)
print({s: (len(DATA[s].sid), DATA[s].n) for s in SPLITS})
print(f"training: {len(TRAIN_SESS):,} sessions, {len(TRAIN_ROWS):,} candidates, {BATCHES_PER_EPOCH:,} batches per epoch")


def epoch_batches(epoch):
    perm = np.random.default_rng(SEED + 1000 + epoch).permutation(TRAIN_ROWS)
    return [perm[i:i + BATCH_SIZE] for i in range(0, len(perm), BATCH_SIZE)]

{'train': (136940, 12279873), 'val': (17117, 1536651), 'test': (17118, 1531525)}
training: 136,940 sessions, 12,279,873 candidates, 5,997 batches per epoch


## Step 5 — The literal BST model

| Part | What it does |
|---|---|
| History tokens | For each of the `HIST_LEN` most recent actions: the sum of product, brand, shop, category, action-type, time-bucket and recency-position embeddings (padding beyond the history's length is masked) |
| Candidate token | The candidate's product, brand, shop and category embeddings, plus a learned *candidate* type embedding and its own position embedding (index `HIST_LEN`), **appended after the history** |
| Transformer encoder | `N_LAYERS` layers of multi-head self-attention over all `HIST_LEN + 1` tokens, with no causal mask: the candidate attends to the whole history, and the history is re-encoded together with this candidate |
| Readout | The encoder output at the candidate's position |
| MLP | Input: readout, candidate embedding, the 23 context features (plus missing-value indicators) and the session's query vector. Output: one logit; sigmoid(logit) = predicted probability of engagement |

Product embeddings start from the item2vec vectors, as per step_3_person_match_feature_engineering.ipynb's feature engineering.

In [ ]:
class LiteralBST(nn.Module):
    def __init__(self, vocab, d, heads, layers, hist_len, ctx_dim, q_dim, dropout, mlp_hidden, item_init=None):
        super().__init__()
        self.item = nn.Embedding(vocab["item"], d, padding_idx=0)
        self.brand = nn.Embedding(vocab["brand"], d, padding_idx=0)
        self.shop = nn.Embedding(vocab["shop"], d, padding_idx=0)
        self.cate = nn.Embedding(vocab["cate3"], d, padding_idx=0)
        self.atype = nn.Embedding(vocab["type"] + 1, d, padding_idx=0)     # last index = the candidate token's type
        self.tbucket = nn.Embedding(vocab["tbucket"], d, padding_idx=0)
        self.pos = nn.Embedding(hist_len + 1, d)                          # 0..L-1 history (most recent first), L = candidate
        self.cand_type = vocab["type"]
        for emb in [self.item, self.brand, self.shop, self.cate, self.atype, self.tbucket, self.pos]:
            nn.init.normal_(emb.weight, std=0.02)
        if item_init is not None:
            w = torch.as_tensor(np.asarray(item_init), dtype=torch.float32)
            if w.shape[1] != d:
                w = w @ torch.randn(w.shape[1], d) / math.sqrt(w.shape[1])
            with torch.no_grad():
                self.item.weight[2:2 + len(w)] = w * 0.1
        with torch.no_grad():
            for emb in [self.item, self.brand, self.shop, self.cate, self.atype, self.tbucket]:
                emb.weight[0].zero_()
        self.norm = nn.LayerNorm(d)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        dims = [2 * d + ctx_dim + q_dim, *mlp_hidden]
        mlp = []
        for a, b in zip(dims[:-1], dims[1:]):
            mlp += [nn.Linear(a, b), nn.ReLU(), nn.Dropout(dropout)]
        self.mlp = nn.Sequential(*mlp[:-1], nn.Linear(dims[-1], 1))

    def forward(self, T):
        B, L = T["h_item"].shape
        dev = T["h_item"].device
        pad = T["h_item"].eq(0)
        pos = torch.arange(L, device=dev).unsqueeze(0)
        h = (self.item(T["h_item"]) + self.brand(T["h_brand"]) + self.shop(T["h_shop"]) + self.cate(T["h_cate3"])
             + self.atype(T["h_type"]) + self.tbucket(T["h_tbucket"]) + self.pos(pos))
        c = (self.item(T["c_item"]) + self.brand(T["c_brand"]) + self.shop(T["c_shop"]) + self.cate(T["c_cate3"])
             + self.atype.weight[self.cand_type] + self.pos.weight[L])            # [B, d]
        seq = torch.cat([h, c.unsqueeze(1)], dim=1)                            # candidate appended to the history
        mask = torch.cat([pad, torch.zeros(B, 1, dtype=torch.bool, device=dev)], dim=1)
        out = self.encoder(self.norm(seq), src_key_padding_mask=mask)
        target = out[:, -1, :]                                                 # representation at the candidate's position
        x = torch.cat([target, c, T["ctx"], T["qvec"]], dim=-1)
        return self.mlp(x).squeeze(-1)                                         # logit; sigmoid = engagement probability


GAIN = torch.tensor([0.0, 1.0, 3.0, 7.0], device=DEVICE)


def pointwise_loss(logits, label):
    y = (label > 0).float()
    w = torch.where(label > 0, GAIN[label.clamp(0, 3)], torch.ones_like(y)) if GRADED_WEIGHTS else torch.ones_like(y)
    return (Fnn.binary_cross_entropy_with_logits(logits.float(), y, reduction="none") * w).sum() / w.sum()


model = LiteralBST(VOCAB, D_MODEL, N_HEADS, N_LAYERS, HIST_LEN, max(CTX_DIM, 1), T2V_TERMS.shape[1],
                   DROPOUT, MLP_HIDDEN, item_init=I2V_VECS).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

parameters: 83,127,169


/tmp/ipykernel_4251/3730850580.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  w = torch.as_tensor(np.asarray(item_init), dtype=torch.float32)


### Evaluation functions
For each session these metrics compute binary NDCG@10 (relevance = label > 0, as in step_2_query_popularity_baseline_models.ipynb), graded NDCG@10 (gains 2^label − 1 = 0, 1, 3, 7), the reciprocal rank and P@1, together with bootstrap confidence intervals over sessions.

They are used for preliminary evaluation of the BST model only and to check the runs.

In [ ]:
N_BOOT = 1000


def per_session_metrics(session_id, score, label, k=10, seed=42):
    # Per-session NDCG@k (binary and graded), reciprocal rank and P@1; step_2_query_popularity_baseline_models.ipynb's ordering and tie-breaking.
    np.random.seed(seed); tb = np.random.rand(len(score))
    s = np.where(np.isnan(score), -np.inf, score)
    order = np.lexsort((-tb, -s, session_id))
    sid, lab = session_id[order], label[order].astype(np.int64)
    rel = lab > 0
    first = np.r_[True, sid[1:] != sid[:-1]]; grp = np.cumsum(first) - 1
    rank = np.arange(len(sid)) - np.flatnonzero(first)[grp] + 1
    disc = np.where(rank <= k, 1 / np.log2(rank + 1), 0.0)
    n_g = grp[-1] + 1
    dcg_b = np.bincount(grp, weights=rel * disc, minlength=n_g)
    gain = 2.0 ** lab - 1
    dcg_g = np.bincount(grp, weights=gain * disc, minlength=n_g)
    n_rel = np.bincount(grp, weights=rel, minlength=n_g)
    cum = np.r_[0, np.cumsum(1 / np.log2(np.arange(2, k + 2)))]
    idcg_b = cum[np.minimum(n_rel, k).astype(int)]
    # ideal graded DCG: gains sorted in decreasing order within each session
    o2 = np.lexsort((-gain, grp))
    r2 = np.arange(len(grp)) - np.flatnonzero(np.r_[True, grp[o2][1:] != grp[o2][:-1]])[np.cumsum(np.r_[True, grp[o2][1:] != grp[o2][:-1]]) - 1] + 1
    idcg_g = np.bincount(grp[o2], weights=gain[o2] * np.where(r2 <= k, 1 / np.log2(r2 + 1), 0.0), minlength=n_g)
    best = np.full(n_g, np.inf); np.minimum.at(best, grp[rel], rank[rel])
    has = n_rel > 0
    sess = sid[first]
    return pd.DataFrame({"session_id": sess[has], "NDCG@10": dcg_b[has] / idcg_b[has],
                         "NDCG@10 graded": dcg_g[has] / idcg_g[has], "MRR": 1 / best[has], "P@1": (best[has] == 1).astype(float)})


def bootstrap_ci(values, n_boot=N_BOOT, seed=SEED):
    values = np.asarray(values, dtype=np.float64)
    idx = np.random.default_rng(seed).integers(0, len(values), size=(n_boot, len(values)))
    means = values[idx].mean(axis=1)
    return values.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


def summarise(per_sess, label=""):
    rows = {}
    for m in ["NDCG@10", "NDCG@10 graded", "MRR", "P@1"]:
        mean, lo, hi = bootstrap_ci(per_sess[m].to_numpy())
        rows[m] = f"{mean:.4f} [{lo:.4f}, {hi:.4f}]"
    rows["sessions"] = len(per_sess)
    return pd.Series(rows, name=label)

## Step 6 — Full training with early stopping and mid-epoch checkpoints

**What this step does.** It trains with AdamW (mixed precision on a GPU), one shuffled pass over all training candidates per epoch.

- **During each epoch.** It prints the running loss, the measured speed and the estimated time left in the epoch. Every `CHECKPOINT_EVERY` batches it saves `bst_literal/checkpoint_last.pt`: the model, the optimiser, and the epoch and batch reached.
- **After each epoch.** It scores all validation candidates and computes graded and binary NDCG@10, MRR and P@1. It keeps the model with the highest validation graded NDCG@10 as `bst_literal/best_model.pt`.

Training stops after `PATIENCE` epochs without improvement, or at `MAX_EPOCHS`. With `RESUME = True`, rerunning the notebook continues from the last checkpoint, inside the epoch if it was interrupted mid-way. Delete `checkpoint_last.pt` to start afresh.

In [ ]:
@torch.no_grad()
def predict(split):
    model.eval()
    D = DATA[split]
    scores = np.zeros(D.n, np.float32)
    for i in range(0, D.n, EVAL_BATCH_SIZE):
        rows = np.arange(i, min(i + EVAL_BATCH_SIZE, D.n))
        T, rows = D.tensors(rows)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            scores[rows] = model(T).float().cpu().numpy()
    return pd.DataFrame({"session_id": np.repeat(np.asarray(D.sid), D.size), "wid": np.asarray(D.C["wid"]),
                         "label": np.asarray(D.C["label"]), "score": scores})


def validation_metrics():
    v = predict("val")
    per = per_session_metrics(v["session_id"].to_numpy(), v["score"].to_numpy(), v["label"].to_numpy())
    return {m: float(per[m].mean()) for m in ["NDCG@10 graded", "NDCG@10", "MRR", "P@1"]}


opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
CKPT, BEST = OUT_DIR / "checkpoint_last.pt", OUT_DIR / "best_model.pt"
state = {"epoch": 0, "step": 0, "best": -1.0, "bad_epochs": 0, "history": [], "loss_sum": 0.0, "loss_n": 0}
if RESUME and CKPT.exists():
    ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); state = ck["state"]
    print(f"resumed at epoch {state['epoch'] + 1}, batch {state['step']:,} (best validation graded NDCG@10 {state['best']:.4f})")


def save_checkpoint():
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "state": state}, CKPT)


while state["epoch"] < MAX_EPOCHS and state["bad_epochs"] < PATIENCE:
    model.train()
    batches = epoch_batches(state["epoch"])
    t0, step0 = time.time(), state["step"]
    for step in range(state["step"], len(batches)):
        T, _ = DATA["train"].tensors(batches[step])
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(T)
        loss = pointwise_loss(logits, T["label"])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        state["loss_sum"] += loss.item(); state["loss_n"] += 1
        state["step"] = step + 1
        done = step + 1 - step0
        if done % 500 == 0 or step + 1 == len(batches):
            rate = done / (time.time() - t0)
            print(f"  epoch {state['epoch'] + 1} batch {step + 1:,}/{len(batches):,}: loss {state['loss_sum'] / state['loss_n']:.4f} | "
                  f"{rate * BATCH_SIZE:,.0f} candidates/s | ~{(len(batches) - step - 1) / rate / 60:.0f} min left in epoch")
        if (step + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint()
    m = validation_metrics()
    state["epoch"] += 1
    state["history"].append({"epoch": state["epoch"], "train_loss": state["loss_sum"] / max(state["loss_n"], 1),
                             **{f"val {k}": v for k, v in m.items()}, "minutes": round((time.time() - t0) / 60, 1)})
    state["step"], state["loss_sum"], state["loss_n"] = 0, 0.0, 0
    if m["NDCG@10 graded"] > state["best"]:
        state["best"], state["bad_epochs"] = m["NDCG@10 graded"], 0
        torch.save(model.state_dict(), BEST)
    else:
        state["bad_epochs"] += 1
    save_checkpoint()
    print(f"epoch {state['epoch']}: val {({k: round(v, 4) for k, v in m.items()})}"
          f"{' | best so far' if state['bad_epochs'] == 0 else ''}")
display(pd.DataFrame(state["history"]))

resumed at epoch 2, batch 0 (best validation graded NDCG@10 0.3868)
  epoch 2 batch 500/5,997: loss 0.1191 | 24,530 candidates/s | ~8 min left in epoch
  epoch 2 batch 1,000/5,997: loss 0.1177 | 24,994 candidates/s | ~7 min left in epoch
  epoch 2 batch 1,500/5,997: loss 0.1177 | 25,135 candidates/s | ~6 min left in epoch
  epoch 2 batch 2,000/5,997: loss 0.1180 | 25,210 candidates/s | ~5 min left in epoch
  epoch 2 batch 2,500/5,997: loss 0.1184 | 24,925 candidates/s | ~5 min left in epoch
  epoch 2 batch 3,000/5,997: loss 0.1189 | 24,997 candidates/s | ~4 min left in epoch
  epoch 2 batch 3,500/5,997: loss 0.1194 | 25,048 candidates/s | ~3 min left in epoch
  epoch 2 batch 4,000/5,997: loss 0.1198 | 25,092 candidates/s | ~3 min left in epoch
  epoch 2 batch 4,500/5,997: loss 0.1201 | 24,958 candidates/s | ~2 min left in epoch
  epoch 2 batch 5,000/5,997: loss 0.1203 | 25,002 candidates/s | ~1 min left in epoch
  epoch 2 batch 5,500/5,997: loss 0.1205 | 25,036 candidates/s | ~1 min le

,epoch,train_loss,val NDCG@10 graded,val NDCG@10,val MRR,val P@1,minutes
0,1,0.136935,0.386816,0.392498,0.394394,0.255886,10.3
1,2,0.120732,0.351074,0.355742,0.357496,0.221359,8.4
2,3,0.095015,0.298594,0.302715,0.302363,0.172811,8.3


## Step 7 — Validation and test results

**What this step does.** It loads the best model (by validation graded NDCG@10), scores the validation and test splits, and saves:
- `bst_literal/val_predictions.parquet` and `test_predictions.parquet` (`session_id`, `wid`, `label`, `score`), in the row order of the feature cache;
- `bst_literal/metrics.json`.

The test split is used only here, once, after training.

In [ ]:
model.load_state_dict(torch.load(BEST, map_location=DEVICE))
PER_SESSION, results = {}, {}
for split in ["val", "test"]:
    pred = predict(split)
    pred.to_parquet(OUT_DIR / f"{split}_predictions.parquet", index=False)
    PER_SESSION[split] = per_session_metrics(pred["session_id"].to_numpy(), pred["score"].to_numpy(), pred["label"].to_numpy())
    results[split] = {m: float(PER_SESSION[split][m].mean()) for m in ["NDCG@10 graded", "NDCG@10", "MRR", "P@1"]}
json.dump({"results": results, "history": state["history"], "context_features": CTX,
           "settings": {"HIST_LEN": HIST_LEN, "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS,
                        "BATCH_SIZE": BATCH_SIZE, "GRADED_WEIGHTS": GRADED_WEIGHTS, "TRAIN_SESSION_FRACTION": TRAIN_SESSION_FRACTION}},
          open(OUT_DIR / "metrics.json", "w"), indent=1)
display(pd.DataFrame(results).T.round(4))

,NDCG@10 graded,NDCG@10,MRR,P@1
val,0.3868,0.3925,0.3944,0.2559
test,0.3847,0.3895,0.3941,0.2576


## Step 8 — Performance tests

Performance tests on the BST model:
- **8.1** — metrics with 95% bootstrap confidence intervals over sessions;
- **8.2** — test results by session type (exact history match, brand or shop match only, no match) and by history length;
- **8.3** — training and inference speed, and peak memory.

There are no ablations here, because each run of literal BST is expensive.

### 8.1 Metrics with 95% confidence intervals

In [ ]:
ci_table = pd.DataFrame([summarise(PER_SESSION[s], s) for s in ["val", "test"]])
ci_table.to_csv(OUT_DIR / "performance_ci.csv")
display(ci_table)

,NDCG@10,NDCG@10 graded,MRR,P@1,sessions
val,"0.3925 [0.3872, 0.3976]","0.3868 [0.3816, 0.3919]","0.3944 [0.3889, 0.3997]","0.2559 [0.2493, 0.2624]",17117
test,"0.3895 [0.3842, 0.3948]","0.3847 [0.3794, 0.3901]","0.3941 [0.3883, 0.3996]","0.2576 [0.2506, 0.2633]",17118


### 8.2 Breakdown by session type

Test sessions are grouped by match type and by history length, using the full cleaned history from the step_3_person_match_feature_engineering.ipynb session chunks.

In [ ]:
def session_types(split):
    ids = set(sessions.loc[sessions["split"] == split, "session_id"])
    rows = []
    for i in range(N_CHUNKS):
        c = pd.read_parquet(STAGE / f"cand_{i:03d}.parquet", columns=["session_id", "wid"])
        c = c[c["session_id"].isin(ids)]
        if not len(c):
            continue
        h = pd.read_parquet(STAGE / f"hist_{i:03d}.parquet", columns=["session_id", "wid"])
        h = h[h["session_id"].isin(ids)]
        exact = c.merge(h.drop_duplicates(), on=["session_id", "wid"])["session_id"].unique()
        ca, ha = np.asarray(PROD_ATTR[prod_rows(c["wid"].to_numpy())]), np.asarray(PROD_ATTR[prod_rows(h["wid"].to_numpy())])
        attr = set()
        for j in [BRAND, SHOP]:
            cc = pd.DataFrame({"session_id": c["session_id"].to_numpy(), "v": ca[:, j]}); cc = cc[cc["v"] != -1]
            hh = pd.DataFrame({"session_id": h["session_id"].to_numpy(), "v": ha[:, j]}).drop_duplicates(); hh = hh[hh["v"] != -1]
            attr |= set(cc.merge(hh, on=["session_id", "v"])["session_id"].unique())
        hl = h.groupby("session_id").size()
        for s in c["session_id"].unique():
            rows.append({"session_id": s, "match": "exact match" if s in set(exact) else ("brand or shop match only" if s in attr else "no match"),
                         "history_len": int(hl.get(s, 0))})
    t = pd.DataFrame(rows)
    t["history"] = pd.cut(t["history_len"], [-1, 49, 199, np.inf], labels=["< 50 actions", "50-199 actions", ">= 200 actions"])
    return t


types_test = session_types("test")
ps = PER_SESSION["test"].merge(types_test, on="session_id")
by_match = pd.DataFrame([summarise(g, k) for k, g in ps.groupby("match")])
by_hist = pd.DataFrame([summarise(g, str(k)) for k, g in ps.groupby("history", observed=True)])
by_match.to_csv(OUT_DIR / "performance_by_match.csv"); by_hist.to_csv(OUT_DIR / "performance_by_history.csv")
print("Test sessions by match type:"); display(by_match)
print("Test sessions by history length:"); display(by_hist)

Test sessions by match type:


,NDCG@10,NDCG@10 graded,MRR,P@1,sessions
brand or shop match only,"0.3415 [0.3331, 0.3499]","0.3360 [0.3277, 0.3444]","0.3409 [0.3321, 0.3499]","0.2022 [0.1922, 0.2123]",6176
exact match,"0.4907 [0.4805, 0.5010]","0.4867 [0.4757, 0.4971]","0.5135 [0.5020, 0.5246]","0.3796 [0.3655, 0.3930]",5089
no match,"0.3521 [0.3438, 0.3610]","0.3473 [0.3389, 0.3561]","0.3465 [0.3370, 0.3562]","0.2098 [0.1992, 0.2201]",5853


Test sessions by history length:


,NDCG@10,NDCG@10 graded,MRR,P@1,sessions
< 50 actions,"0.3816 [0.3723, 0.3900]","0.3784 [0.3690, 0.3870]","0.3885 [0.3785, 0.3978]","0.2549 [0.2436, 0.2656]",6325
50-199 actions,"0.3928 [0.3843, 0.4018]","0.3864 [0.3781, 0.3951]","0.3949 [0.3858, 0.4043]","0.2591 [0.2486, 0.2698]",6747
>= 200 actions,"0.3962 [0.3853, 0.4063]","0.3915 [0.3802, 0.4020]","0.4017 [0.3904, 0.4126]","0.2593 [0.2459, 0.2726]",4046


### 8.3 Speed and memory

Throughput over 50 training batches (including the backward pass) and over the full validation split, together with peak GPU memory and process RAM. Report these with the runtime type and `BATCH_SIZE`.

In [ ]:
def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
bench = LiteralBST(VOCAB, D_MODEL, N_HEADS, N_LAYERS, HIST_LEN, max(CTX_DIM, 1), T2V_TERMS.shape[1],
                   DROPOUT, MLP_HIDDEN, item_init=I2V_VECS).to(DEVICE)
bench.load_state_dict(model.state_dict())
o = torch.optim.AdamW(bench.parameters(), lr=LEARNING_RATE)
sc = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
batches = epoch_batches(999)[:50]
bench.train(); sync(); t0 = time.time()
for rows in batches:
    T, _ = DATA["train"].tensors(rows)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        loss = pointwise_loss(bench(T), T["label"])
    o.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.step(o); sc.update()
sync(); t_train = time.time() - t0
n_c = sum(len(b) for b in batches)
model.eval(); sync(); t0 = time.time()
_ = predict("val")
sync(); t_inf = time.time() - t0
perf = pd.Series({
    "device": str(DEVICE) + (f" ({torch.cuda.get_device_name()})" if DEVICE.type == "cuda" else ""),
    "BATCH_SIZE": BATCH_SIZE, "sequence length (HIST_LEN + 1)": HIST_LEN + 1,
    "training: candidates / s": round(n_c / t_train),
    "inference: candidates / s": round(DATA["val"].n / t_inf),
    "estimated minutes per training epoch": round(len(TRAIN_ROWS) / (n_c / t_train) / 60, 1),
    "peak GPU memory (GB)": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else "n/a (CPU)",
    "process RAM (GB)": round(psutil.Process().memory_info().rss / 1e9, 2) if "psutil" in globals() else "n/a",
}, name="value")
perf.to_csv(OUT_DIR / "performance_speed.csv")
display(perf.to_frame())
del bench, o

,value
device,cuda (NVIDIA L4)
BATCH_SIZE,2048
sequence length (HIST_LEN + 1),51
training: candidates / s,28389
inference: candidates / s,139857
estimated minutes per training epoch,7.2
peak GPU memory (GB),4.12
process RAM (GB),5.11


## Notes

- **Comparing with LightGBM.** Both model training use the same 23 features, 80/10/10 split, validation and test sessions, and graded NDCG@10 with gains 0, 1, 3, 7 as the selection metric, so the validation results are directly comparable. That notebook reports a validation graded NDCG@10 of 0.4054 for its best LightGBM. Its NDCG function breaks ties by the original row order, and this one breaks them at random. Ties are rare with continuous scores, so the difference is negligible.
- **Changing settings.** Delete `bst_literal/checkpoint_last.pt` before retraining with different settings. After changing `HIST_LEN`, also delete `/content/part8_local/assembled`, since the inputs must be reassembled.

## Step 9 — Training predictions for the shared results table

Loads the best model saved in Step 5 and scores the **training** split, writing `bst_literal/train_predictions.parquet` in the same format as the validation and test files. The results table in `step_8_results_and_experiments.ipynbb` (Section 6) scores training and test users for every model, so the BST needs a training file as well. No training happens here; the model is only applied.

In [ ]:
@torch.no_grad()
def predict(split):
    model.eval()
    D = DATA[split]
    scores = np.zeros(D.n, np.float32)
    for i in range(0, D.n, EVAL_BATCH_SIZE):
        rows = np.arange(i, min(i + EVAL_BATCH_SIZE, D.n))
        T, rows = D.tensors(rows)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            scores[rows] = model(T).float().cpu().numpy()
    return pd.DataFrame({"session_id": np.repeat(np.asarray(D.sid), D.size), "wid": np.asarray(D.C["wid"]),
                         "label": np.asarray(D.C["label"]), "score": scores})

BEST = OUT_DIR / "best_model.pt"
model.load_state_dict(torch.load(BEST, map_location=DEVICE))
predict("train").to_parquet(OUT_DIR / "train_predictions.parquet", index=False)
print("saved", OUT_DIR / "train_predictions.parquet")

saved /content/drive/MyDrive/BT4222 Final/JDSearch/processed/bst_literal/train_predictions.parquet


## Step 10 — Feature-family experiment on BST

Runs the feature-family experiment of `step_8_results_and_experiments.ipynbb` (Sections 7.3 and 7.4) on the main model. For each feature family, a fresh BST is trained with that family's columns zeroed out of the context input (the features and their missing-value indicators), so the model cannot use them. Everything else is the setup of this notebook: same architecture, loss, optimiser and early stopping on validation graded NDCG@10. Each of the twelve runs trains on the same random quarter of the training sessions and is scored on all validation sessions, as in the LightGBM experiment. The validation predictions of each run's best epoch are saved to `processed/bst_experiments/`, where Section 7.2 of `step_8_results_and_experiments.ipynbb` scores them with the shared scoring code. Nothing in `bst_literal/` is changed.

In [ ]:
# Uses the data, model class and evaluation functions defined in Steps 0 to 5. Does not change bst_literal/.
EXP_DIR = OUTPUT_DIR / "bst_experiments"; EXP_DIR.mkdir(parents=True, exist_ok=True)
EXP_TRAIN_FRACTION = 0.25            # same idea as the LightGBM experiments: a quarter of the training sessions
EXP_MAX_EPOCHS, EXP_PATIENCE = 6, 2  # early stopping on validation graded NDCG@10
EXP_SEED = 4222
EXP_FILE = EXP_DIR / "bst_family_experiment_runs.csv"

# Families as defined in the feature dictionary (same grouping as step_8_results_and_experiments.ipynbb, Section 7.3)
FAMILIES = {
    "A": ("Exact product history", ["exact_wid_last_seen_days"]),
    "B": ("Brand / shop / category affinity", ["brand_aff", "cate3_aff", "shop_share", "shop_last_seen_days"]),
    "C": ("Title similarity to history", ["title_cos_max", "title_cos_profile"]),
    "D": ("item2vec similarity", ["i2v_cos_mean"]),
    "E": ("Similarity to the last action", ["last_same_brand", "last_same_cate3", "last_i2v_cos"]),
    "F": ("History under the same query", ["qhist_wid_cnt"]),
    "G": ("Popularity priors (training users only)", ["item_ctr_train", "cate3_ctr_train"]),
    "H": ("Relative to the other candidates in the slate", ["cate3_slate_share", "shop_aff_slate_pct"]),
    "Q": ("Current-query match", ["name_query_match", "brand_name_query_match", "cate_name_1_query_match",
                                 "cate_name_2_query_match", "cate_name_3_query_match", "cate_name_4_query_match",
                                 "query_match_count"]),
}
assert sorted(f for _, fs in FAMILIES.values() for f in fs) == sorted(CTX), "family lists do not match the 23 features"
PERSONAL = [f for k in "ABCDEF" for f in FAMILIES[k][1]] + ["shop_aff_slate_pct"]   # same definition as Section 7.4

RUNS = [("All features", [])] + \
       [(f"Without {k}: {name}", feats) for k, (name, feats) in FAMILIES.items()] + \
       [("Without all personal history (A-F)", PERSONAL)]

# the training quarter: fixed by EXP_SEED so every run uses the same sessions
rng_exp = np.random.default_rng(EXP_SEED)
EXP_SESS = np.sort(rng_exp.choice(n_tr_sess, size=int(n_tr_sess * EXP_TRAIN_FRACTION), replace=False))
EXP_ROWS = DATA["train"].rows_of_sessions(EXP_SESS)
print(f"experiment training sample: {len(EXP_SESS):,} sessions, {len(EXP_ROWS):,} candidates")


def ctx_mask(drop):
    # 1 for columns the model may use, 0 for the dropped features and their missing-value indicators
    m = np.ones(CTX_DIM, np.float32)
    for j, f in enumerate(CTX):
        if f in drop: m[j] = 0.0
    for j, f in enumerate(CTX_MISS):
        if f in drop: m[len(CTX) + j] = 0.0
    return torch.tensor(m, device=DEVICE)


@torch.no_grad()
def predict_masked(m, split, mask):
    m.eval(); D = DATA[split]; scores = np.zeros(D.n, np.float32)
    for i in range(0, D.n, EVAL_BATCH_SIZE):
        rows = np.arange(i, min(i + EVAL_BATCH_SIZE, D.n))
        T, rows = D.tensors(rows); T["ctx"] = T["ctx"] * mask
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            scores[rows] = m(T).float().cpu().numpy()
    return pd.DataFrame({"session_id": np.repeat(np.asarray(D.sid), D.size), "wid": np.asarray(D.C["wid"]),
                         "label": np.asarray(D.C["label"]), "score": scores})


def run_variant(name, drop):
    fname = "".join(ch if ch.isalnum() else "_" for ch in name).strip("_")
    out_pred = EXP_DIR / f"{fname}_val_predictions.parquet"
    if out_pred.exists():
        print("already done, skipping:", name); return
    print(f"\n--- {name}: {len(CTX) - len(drop)} features")
    mask = ctx_mask(drop)
    torch.manual_seed(EXP_SEED); np.random.seed(EXP_SEED)
    m = LiteralBST(VOCAB, D_MODEL, N_HEADS, N_LAYERS, HIST_LEN, max(CTX_DIM, 1), T2V_TERMS.shape[1],
                   DROPOUT, MLP_HIDDEN, item_init=I2V_VECS).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sc = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best, best_epoch, bad, best_pred, hist = -1.0, 0, 0, None, []
    for epoch in range(EXP_MAX_EPOCHS):
        m.train(); t0 = time.time(); loss_sum = 0.0; n = 0
        perm = np.random.default_rng(EXP_SEED + 1000 + epoch).permutation(EXP_ROWS)
        for b in range(0, len(perm), BATCH_SIZE):
            T, _ = DATA["train"].tensors(perm[b:b + BATCH_SIZE]); T["ctx"] = T["ctx"] * mask
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                logits = m(T)
            loss = pointwise_loss(logits, T["label"])
            opt.zero_grad(set_to_none=True); sc.scale(loss).backward()
            sc.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 1.0); sc.step(opt); sc.update()
            loss_sum += loss.item(); n += 1
        v = predict_masked(m, "val", mask)
        per = per_session_metrics(v["session_id"].to_numpy(), v["score"].to_numpy(), v["label"].to_numpy())
        g = float(per["NDCG@10 graded"].mean())
        hist.append({"epoch": epoch + 1, "train_loss": loss_sum / max(n, 1), "val NDCG@10 graded": g,
                     "val MRR": float(per["MRR"].mean()), "val P@1": float(per["P@1"].mean()),
                     "minutes": round((time.time() - t0) / 60, 1)})
        print(f"  epoch {epoch + 1}: val graded NDCG@10 {g:.4f} | {hist[-1]['minutes']} min")
        if g > best:
            best, best_epoch, bad, best_pred = g, epoch + 1, 0, v
        else:
            bad += 1
            if bad >= EXP_PATIENCE: break
    best_pred.to_parquet(out_pred, index=False)
    row = {"model": name, "features": len(CTX) - len(drop), "epochs_run": len(hist), "best_epoch": best_epoch,
           "val NDCG@10 graded": round(best, 4),
           "val MRR": round(hist[best_epoch - 1]["val MRR"], 4), "val P@1": round(hist[best_epoch - 1]["val P@1"], 4),
           "minutes": round(sum(h["minutes"] for h in hist), 1), "predictions": out_pred.name}
    done = pd.read_csv(EXP_FILE) if EXP_FILE.exists() else pd.DataFrame()
    pd.concat([done, pd.DataFrame([row])], ignore_index=True).to_csv(EXP_FILE, index=False)
    print(f"  saved: best epoch {best_epoch}, val graded NDCG@10 {best:.4f}")
    del m, opt, sc, best_pred; gc.collect(); torch.cuda.empty_cache()


for name, drop in RUNS:
    run_variant(name, drop)

exp_table = pd.read_csv(EXP_FILE).set_index("model")
exp_table["NDCG@10 drop"] = exp_table.loc["All features", "val NDCG@10 graded"] - exp_table["val NDCG@10 graded"]
display(exp_table.round(4))

experiment training sample: 34,235 sessions, 3,070,732 candidates

--- All features: 23 features
  epoch 1: val graded NDCG@10 0.3820 | 2.2 min
  epoch 2: val graded NDCG@10 0.3297 | 2.2 min
  epoch 3: val graded NDCG@10 0.2506 | 2.2 min
  saved: best epoch 1, val graded NDCG@10 0.3820

--- Without A: Exact product history: 22 features
  epoch 1: val graded NDCG@10 0.3795 | 2.2 min
  epoch 2: val graded NDCG@10 0.3198 | 2.2 min
  epoch 3: val graded NDCG@10 0.2444 | 2.2 min
  saved: best epoch 1, val graded NDCG@10 0.3795

--- Without B: Brand / shop / category affinity: 19 features
  epoch 1: val graded NDCG@10 0.3810 | 2.2 min
  epoch 2: val graded NDCG@10 0.3239 | 2.2 min
  epoch 3: val graded NDCG@10 0.2417 | 2.2 min
  saved: best epoch 1, val graded NDCG@10 0.3810

--- Without C: Title similarity to history: 21 features
  epoch 1: val graded NDCG@10 0.3748 | 2.2 min
  epoch 2: val graded NDCG@10 0.3176 | 2.2 min
  epoch 3: val graded NDCG@10 0.2429 | 2.2 min
  saved: best epoch 1,

,features,epochs_run,best_epoch,val NDCG@10 graded,val MRR,val P@1,minutes,predictions,NDCG@10 drop
model,,,,,,,,,
All features,23,3,1,0.3820,0.3893,0.2530,6.6,All_features_val_predictions.parquet,0.0000
Without A: Exact product history,22,3,1,0.3795,0.3853,0.2476,6.6,Without_A__Exact_product_history_val_predictio...,0.0025
Without B: Brand / shop / category affinity,19,3,1,0.3810,0.3884,0.2518,6.6,Without_B__Brand___shop___category_affinity_va...,0.0010
Without C: Title similarity to history,21,3,1,0.3748,0.3817,0.2439,6.6,Without_C__Title_similarity_to_history_val_pre...,0.0072
Without D: item2vec similarity,22,3,1,0.3781,0.3850,0.2474,6.6,Without_D__item2vec_similarity_val_predictions...,0.0039
Without E: Similarity to the last action,20,3,1,0.3794,0.3858,0.2477,6.6,Without_E__Similarity_to_the_last_action_val_p...,0.0026
Without F: History under the same query,22,3,1,0.3810,0.3882,0.2503,6.6,Without_F__History_under_the_same_query_val_pr...,0.0010
Without G: Popularity priors (training users only),21,3,1,0.3513,0.3577,0.2233,6.6,Without_G__Popularity_priors__training_users_o...,0.0307
Without H: Relative to the other candidates in the slate,21,3,1,0.3791,0.3856,0.2493,6.6,Without_H__Relative_to_the_other_candidates_in...,0.0029


**Result:** With all features the quarter-sample BST reaches 0.382 validation graded NDCG@10. Removing all personal-history features costs 0.057, removing the popularity family 0.030, and removing any single family at most 0.007; the query-match family costs 0.005. Every run reached its best score after the first epoch and declined from the second, the same pattern as the training-to-test gap of the full model. The scored table with every metric is in Section 7.2 of `step_8_results_and_experiments.ipynbb`.